# 第 2 周末练习 —— 技术问答助手（含工具与语音加分项）

## 练习目标（理念）

用第 2 周所学，把第 1 周的技术问答器做成**完整原型**：

- **Gradio UI**：`gr.Blocks` + `ChatInterface` / 自定义聊天布局
- **流式输出**：生成器 `yield`，边生成边刷新
- **专家 System Prompt**：面向 Python / ML / LLM Engineering 的导师人设
- **模型切换**：GPT、Claude（经 OpenRouter）、本地 Llama（Ollama）
- **加分：Tools**：`execute_python_code`、`search_documentation`
- **加分：语音**：Whisper 转写输入 + OpenAI TTS 朗读输出

## 和本课第 2 周的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|-------------------|
| System Prompt | `SYSTEM_PROMPT` |
| Function Calling | `tools` + `handle_tool_calls` + `chat(..., use_tools)` |
| Streaming | 工具轮结束后再流式/逐字展示回复 |
| 多客户端 | OpenRouter / Anthropic URL / Ollama `localhost` |
| 语音 I/O | `whisper-1` + `tts-1` |

## 本笔记本结构

1. 环境与多模型客户端
2. System Prompt 与工具定义
3. 基础版 Gradio（模型下拉 + 工具开关）
4. 完整版 Gradio（麦克风输入 + TTS 输出）

## 怎么跑

1. `.env` 准备 `OPENAI_API_KEY`（OpenRouter/直连视你的配置）；可选 `ANTHROPIC_API_KEY`
2. 若用本地 Llama：先启动 Ollama 并 pull `llama3.2`
3. 从上到下运行；先会出现基础 UI，最后一格是带语音的完整 UI
4. 可试：解释 `yield from` 代码、让模型 `print([x**2 for x in range(5)])`、搜 `RAG` 文档


In [11]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量（OPENAI_API_KEY、ANTHROPIC_API_KEY 等）
import os
# 标准库 json：解析 tool_call.function.arguments（模型返回 JSON 字符串）
import json
# load_dotenv：把 .env 密钥读进进程环境，避免写进代码
from dotenv import load_dotenv
# OpenAI 客户端：同时用于 OpenRouter / Anthropic 兼容端点 / Ollama / Whisper / TTS
from openai import OpenAI
# Gradio：Blocks、ChatInterface、Audio、Dropdown 等 UI 组件
import gradio as gr


In [12]:
# ========== 环境：加载 API Key 并做存在性检查 ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 读取 OpenAI / OpenRouter 常用的 OPENAI_API_KEY
openai_api_key = os.getenv('OPENAI_API_KEY')
# 可选：Anthropic 直连密钥（没有也能靠 OpenRouter 走 Claude）
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

# 粗检并打印前缀；英文 print 保持原样（排查文案）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional - can use OpenRouter for Claude)")


In [13]:
# ========== 客户端与模型表：OpenRouter / Anthropic / Ollama ==========

# openai_client：经 OpenRouter 统一网关访问多家模型（base_url 勿改）
openai_client = OpenAI(
    api_key=openai_api_key,
    base_url="https://openrouter.ai/api/v1"
)

# Anthropic 官方 OpenAI 兼容 URL；有 key 才创建客户端，否则为 None
anthropic_url = "https://api.anthropic.com/v1/"
anthropic_client = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None

# 本地 Ollama 的 OpenAI 兼容端点；api_key 占位字符串 "ollama" 即可
ollama_url = "http://localhost:11434/v1"
ollama_client = OpenAI(api_key="ollama", base_url=ollama_url)

# MODELS：显示名 → (真正的 model id, 对应 client)
MODELS = {
    "GPT-4.1-mini": ("gpt-4.1-mini", openai_client),
    "GPT-4.1": ("gpt-4.1", openai_client),
    "Claude Sonnet 4.5": ("anthropic/claude-sonnet-4", openai_client),  # via OpenRouter
    "Llama 3.2 (Local)": ("llama3.2", ollama_client),
}

# 打印可选模型列表，方便确认初始化成功
print("Available models:", list(MODELS.keys()))


In [14]:
# ========== System Prompt：技术导师人设（英文指令保留，改译会改行为）==========

# SYSTEM_PROMPT：作为 role=system 的 content，塑造专业范围与教学风格
SYSTEM_PROMPT = """You are an expert technical tutor and programming assistant specializing in:
- Python programming and best practices
- Machine Learning and Deep Learning concepts
- LLM Engineering (transformers, fine-tuning, RAG, agents)
- Software architecture and design patterns
- Data structures and algorithms

Your teaching style:
1. Start with a clear, concise explanation
2. Provide code examples when relevant (using markdown code blocks)
3. Explain the "why" behind concepts, not just the "how"
4. Use analogies to make complex topics accessible
5. If you're uncertain about something, say so honestly

When explaining code, break it down step by step. When asked about errors, 
help debug methodically. Always encourage learning and exploration.

Respond in markdown format for better readability."""


In [15]:
# ========== 工具定义：执行 Python + 模拟文档搜索（Function Calling）==========

def execute_python_code(code: str) -> str:
    """在受限命名空间里执行简单 Python，捕获 stdout；演示用沙箱，非生产安全隔离。"""
    # 调试打印：确认工具被模型调用了
    print(f"Tool called: Executing Python code...")
    try:
        # 极度精简的 __builtins__：只暴露常用内建，降低误用风险（仍非真正安全沙箱）
        namespace = {"__builtins__": {"print": print, "len": len, "range": range, 
                                       "list": list, "dict": dict, "str": str,
                                       "int": int, "float": float, "sum": sum,
                                       "max": max, "min": min, "sorted": sorted,
                                       "enumerate": enumerate, "zip": zip}}
        
        # 把 sys.stdout 临时重定向到 StringIO，以便拿到 print 输出
        import io
        import sys
        old_stdout = sys.stdout
        sys.stdout = captured_output = io.StringIO()
        
        # 在受限 namespace执行用户/模型给的代码字符串
        exec(code, namespace)
        
        # 恢复 stdout 并取出捕获文本
        sys.stdout = old_stdout
        output = captured_output.getvalue()
        
        # 有输出/无输出两种成功文案（字符串保持原样，可能被模型继续阅读）
        return f"Code executed successfully.\nOutput:\n{output}" if output else "Code executed successfully (no output)"
    except Exception as e:
        return f"Error executing code: {str(e)}"


def search_documentation(query: str) -> str:
    """用本地小词典模拟「搜文档」；真实项目可换成向量库或搜索 API。"""
    print(f"Tool called: Searching documentation for '{query}'...")
    
    # 主题关键词 → 说明片段（教学演示数据，英文内容保持原样）
    docs = {
        "transformer": "The Transformer architecture uses self-attention mechanisms to process sequences in parallel. Key components: Multi-Head Attention, Feed-Forward Networks, Positional Encoding. Introduced in 'Attention Is All You Need' (2017).",
        "attention": "Self-attention computes relationships between all positions in a sequence. Formula: Attention(Q,K,V) = softmax(QK^T/√d_k)V. Multi-head attention runs multiple attention operations in parallel.",
        "gradient descent": "Gradient descent optimizes parameters by iteratively moving in the direction of steepest descent. Variants: SGD, Adam, RMSprop. Learning rate is a key hyperparameter.",
        "backpropagation": "Backpropagation computes gradients using the chain rule, propagating errors backward through the network. Essential for training neural networks.",
        "rag": "Retrieval-Augmented Generation (RAG) combines retrieval of relevant documents with LLM generation. Steps: 1) Embed query, 2) Retrieve similar docs, 3) Augment prompt, 4) Generate response.",
        "fine-tuning": "Fine-tuning adapts a pre-trained model to a specific task. Methods: Full fine-tuning, LoRA, QLoRA, Prompt tuning. Trade-off between performance and compute cost.",
    }
    
    # 简单子串匹配：query 里是否包含某个 key
    query_lower = query.lower()
    for key, value in docs.items():
        if key in query_lower:
            return f"Documentation found for '{key}':\n{value}"
    
    return f"No specific documentation found for '{query}'. Try asking about: transformer, attention, gradient descent, backpropagation, RAG, or fine-tuning."


# OpenAI tools schema：type=function + name/description/parameters（英文保留，模型靠它决定是否调用）
tools = [
    {
        "type": "function",
        "function": {
            "name": "execute_python_code",
            "description": "Execute a Python code snippet and return the output. Use this when the user asks to run or test code.",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "The Python code to execute"
                    }
                },
                "required": ["code"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "search_documentation",
            "description": "Search technical documentation for ML/AI concepts. Use this to provide accurate information about specific topics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The topic or concept to search for"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

# 名字 → 可调用的 Python 函数，供 handle_tool_calls 分发
tool_functions = {
    "execute_python_code": execute_python_code,
    "search_documentation": search_documentation
}

print("Tools defined:", [t["function"]["name"] for t in tools])


In [16]:
# ========== handle_tool_calls：把模型的 tool_calls 变成 role=tool 消息 ==========

def handle_tool_calls(message):
    """解析 assistant.message.tool_calls，执行对应本地函数，返回 tool 角色消息列表。"""
    responses = []
    for tool_call in message.tool_calls:
        # 模型要调的函数名
        function_name = tool_call.function.name
        # arguments 是 JSON 字符串 → 字典
        arguments = json.loads(tool_call.function.arguments)
        
        if function_name in tool_functions:
            # 本练习约定每个工具只有一个参数：取 values() 的第一个
            arg_value = list(arguments.values())[0]
            result = tool_functions[function_name](arg_value)
            # OpenAI 协议：role=tool + tool_call_id 把结果接回对话
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
        else:
            # 未知函数名：仍回一条 tool 消息，避免对话链断裂
            responses.append({
                "role": "tool",
                "content": f"Unknown function: {function_name}",
                "tool_call_id": tool_call.id
            })
    return responses


In [17]:
# ========== 核心 chat：模型切换 + 可选 tools + 流式展示 ==========

def chat(message, history, model_name, use_tools):
    """
    Gradio 主回调：
    - 按 model_name 选 MODELS 里的 (model_id, client)
    - 可选 tools；Llama 路径默认关掉工具
    - 先非流式处理 tool_calls 循环，再把最终文本 yield 给 UI
    """
    # 非法模型名：直接 yield 错误文案（英文保持，可能被 UI/用户对照）
    if model_name not in MODELS:
        yield "Error: Invalid model selected"
        return
    
    # 解包：真正的模型 id + 对应 OpenAI 兼容客户端
    model_id, client = MODELS[model_name]
    
    # 拼 messages：system + 历史 + 当前 user
    history_messages = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history_messages + [{"role": "user", "content": message}]
    
    try:
        # Llama 本地路径通常工具支持弱：名字里含 "Llama" 则强制不用 tools
        active_tools = tools if use_tools and "Llama" not in model_name else None
        
        # 第一轮非流式：方便检查 finish_reason 是否为 tool_calls
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=active_tools,
            stream=False  # First call without streaming to check for tool calls
        )
        
        # 空 choices：网关/模型异常时的兜底
        if not response.choices:
            yield "Error: No response received from the model. Please try again."
            return
        
        # 可能多轮 tool：while 直到不再请求工具
        while response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            
            # 防御：标记是 tool_calls 但列表为空则跳出
            if not assistant_message.tool_calls:
                break
                
            tool_responses = handle_tool_calls(assistant_message)
            
            # 把 assistant（含 tool_calls）+ 各 tool 结果追加进 messages
            messages.append(assistant_message)
            messages.extend(tool_responses)
            
            # 带着工具结果再问一轮（仍非流式，可能继续要工具）
            response = client.chat.completions.create(
                model=model_id,
                messages=messages,
                tools=active_tools,
                stream=False
            )
            
            if not response.choices:
                yield "Error: No response after tool call. Please try again."
                return
        
        # 若非流式响应里已有完整 content：逐字符 yield，模拟流式观感
        if response.choices[0].message.content:
            content = response.choices[0].message.content
            result = ""
            for char in content:
                result += char
                yield result
        else:
            # 否则再开真正的 stream=True 请求
            stream = client.chat.completions.create(
                model=model_id,
                messages=messages,
                stream=True
            )
            
            result = ""
            for chunk in stream:
                # 安全取 delta.content（有的 chunk 只有 role / 空 choices）
                if chunk.choices and len(chunk.choices) > 0:
                    delta = chunk.choices[0].delta
                    if delta and delta.content:
                        result += delta.content
                        yield result
            
            # 流式结束仍无字：给出提示
            if not result:
                yield "I received your message but couldn't generate a response. Please try again."
                
    except Exception as e:
        yield f"Error: {str(e)}\n\nMake sure the selected model is available and your API keys are configured."


In [19]:
# ========== 基础版 Gradio：ChatInterface + 模型下拉 + 工具开关 ==========

with gr.Blocks(title="Technical Q&A Assistant") as demo:
    # UI Markdown 文案保持英文原样（展示层，不改可执行逻辑）
    gr.Markdown("""
    # Technical Q&A Assistant
    
    Your AI-powered tutor for Python, Machine Learning, and LLM Engineering.
    
    **Features:**
    - Switch between GPT, Claude, and local Llama models
    - Enable tools for code execution and documentation search
    - Streaming responses for real-time feedback
    """)
    
    # 同一行：模型选择 + 是否启用 tools
    with gr.Row():
        model_selector = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="GPT-4.1-mini",
            label="Select Model"
        )
        tools_toggle = gr.Checkbox(
            value=True,
            label="Enable Tools (code execution & doc search)"
        )
    
    # ChatInterface：fn=chat；additional_inputs 依次对应 model_name、use_tools
    chatbot = gr.ChatInterface(
        fn=chat,
        type="messages",
        additional_inputs=[model_selector, tools_toggle],
        examples=[
            ["Explain what this code does: yield from {book.get('author') for book in books if book.get('author')}"],
            ["What is the transformer architecture and how does attention work?"],
            ["Can you run this code for me: print([x**2 for x in range(5)])"],
            ["Search the documentation for RAG"],
            ["What's the difference between fine-tuning and prompt engineering?"],
        ],
        title=None,  # We have our own title above
    )

# 启动基础版 UI（本机端口；与后面完整版可能各开一个）
demo.launch()


In [20]:
# ========== 加分：语音 —— Whisper 转写 + TTS 朗读 ==========

def transcribe_audio(audio_path):
    """用 OpenAI Whisper 把音频文件转成文字；失败则返回带前缀的错误串。"""
    # 没有录音路径就返回空串
    if audio_path is None:
        return ""
    
    try:
        with open(audio_path, "rb") as audio_file:
            # 直连 OpenAI（默认读环境 OPENAI_API_KEY）；OpenRouter 往往不支持 audio
            direct_openai = OpenAI()  # Uses OPENAI_API_KEY from env
            transcript = direct_openai.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )
            return transcript.text
    except Exception as e:
        return f"[Audio transcription error: {str(e)}]"


def text_to_speech(text):
    """用 OpenAI TTS 把文本合成 mp3 临时文件路径；失败返回 None。"""
    try:
        direct_openai = OpenAI()  # Uses OPENAI_API_KEY from env
        response = direct_openai.audio.speech.create(
            model="tts-1",
            voice="alloy",  # Options: alloy, echo, fable, onyx, nova, shimmer
            input=text[:4096]  # TTS has a character limit
        )
        
        # 写入 NamedTemporaryFile，delete=False 以便 Gradio 还能读到文件
        import tempfile
        with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
            response.stream_to_file(f.name)
            return f.name
    except Exception as e:
        print(f"TTS Error: {e}")
        return None


def chat_with_audio(message, audio_input, history, model_name, use_tools, enable_tts):
    """
    扩展聊天：可把麦克风音频转成 message，再复用 chat() 流式回复；
    结束后可选 TTS，第二次 yield 带上音频路径。
    """
    # 若提供了音频，优先转写并覆盖 message
    if audio_input is not None:
        transcribed = transcribe_audio(audio_input)
        if transcribed:
            message = transcribed
            print(f"Transcribed audio: {message}")
    
    # 既无文字又无有效转写：空响应
    if not message:
        yield "", None
        return
    
    # 复用核心 chat：先流式推文字，音频位先占 None
    full_response = ""
    for partial in chat(message, history, model_name, use_tools):
        full_response = partial
        yield full_response, None
    
    # 需要语音输出时，对完整回复做 TTS
    if enable_tts and full_response:
        audio_output = text_to_speech(full_response)
        yield full_response, audio_output
    else:
        yield full_response, None


In [21]:
# ========== 完整版 Gradio：自定义 Chatbot + 麦克风 + TTS ==========

def process_message(message, audio, history, model_name, use_tools, enable_tts):
    """提交阶段：可选转写音频，把 user 消息追加进 history，并清空输入框。"""
    # 有音频且转写成功（非错误前缀）则用转写文本当 message
    if audio is not None:
        transcribed = transcribe_audio(audio)
        if transcribed and not transcribed.startswith("[Audio"):
            message = transcribed
    
    # 没有有效问题：原样返回 history
    if not message:
        return history, "", None
    
    # 追加用户消息（messages 格式：role/content）
    history = history + [{"role": "user", "content": message}]
    
    # 返回：更新后的 history、清空文本框、音频输出先置空
    return history, "", None


def generate_response(history, model_name, use_tools, enable_tts):
    """生成阶段：对 history 最后一条 user 调用 chat()，流式写回 assistant，可选 TTS。"""
    # 防御：没有对话或最后一条不是 user，就不生成
    if not history or history[-1]["role"] != "user":
        yield history, None
        return
    
    user_message = history[-1]["content"]
    
    # chat() 自己会再拼 user；这里把「最后一条 user」从 history 里摘掉再传入
    prev_history = history[:-1]
    
    # 先占位一条空的 assistant，流式过程中不断改 content
    history = history + [{"role": "assistant", "content": ""}]
    
    full_response = ""
    for partial in chat(user_message, prev_history, model_name, use_tools):
        full_response = partial
        history[-1]["content"] = partial
        yield history, None
    
    # 可选：整段回复转语音
    if enable_tts and full_response:
        audio_path = text_to_speech(full_response)
        yield history, audio_path


# Soft 主题的完整 UI；title 字符串保持原样
with gr.Blocks(title="Technical Q&A Assistant", theme=gr.themes.Soft()) as demo_full:
    gr.Markdown("""
    # Technical Q&A Assistant
    
    Your AI-powered tutor for **Python**, **Machine Learning**, and **LLM Engineering**.
    
    ### Features:
    - **Model Switching**: Choose between GPT, Claude, or local Llama
    - **Tools**: Code execution and documentation search
    - **Voice Input**: Speak your questions (requires OpenAI API)
    - **Voice Output**: Listen to responses (optional TTS)
    - **Streaming**: Real-time response generation
    """)
    
    # 控制条：模型 / 工具 / TTS
    with gr.Row():
        with gr.Column(scale=1):
            model_selector = gr.Dropdown(
                choices=list(MODELS.keys()),
                value="GPT-4.1-mini",
                label="Select Model"
            )
        with gr.Column(scale=1):
            tools_toggle = gr.Checkbox(
                value=True,
                label="Enable Tools"
            )
        with gr.Column(scale=1):
            tts_toggle = gr.Checkbox(
                value=False,
                label="Enable Voice Output"
            )
    
    # 自定义 Chatbot（type=messages 与上面 history 结构一致）
    chatbot = gr.Chatbot(
        label="Conversation",
        height=400,
        type="messages"
    )
    
    # 文本输入 + 麦克风
    with gr.Row():
        with gr.Column(scale=4):
            text_input = gr.Textbox(
                label="Type your question",
                placeholder="Ask about Python, ML, transformers, or any technical topic...",
                lines=2
            )
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                label="Or speak",
                sources=["microphone"],
                type="filepath"
            )
    
    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear Chat")
    
    # 语音回复播放器；autoplay=True 生成后自动播
    audio_output = gr.Audio(
        label="Voice Response",
        autoplay=True,
        visible=True
    )
    
    gr.Markdown("""
    ### Example Questions:
    - "Explain what this code does: `yield from {book.get('author') for book in books}`"
    - "What is the transformer architecture?"
    - "Run this code: `print([x**2 for x in range(5)])`"
    - "Search documentation for RAG"
    """)
    
    # 点击 Send：先 process_message，再 then 链式 generate_response
    submit_btn.click(
        fn=process_message,
        inputs=[text_input, audio_input, chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, text_input, audio_output]
    ).then(
        fn=generate_response,
        inputs=[chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, audio_output]
    )
    
    # 文本框回车：同样的两段链路
    text_input.submit(
        fn=process_message,
        inputs=[text_input, audio_input, chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, text_input, audio_output]
    ).then(
        fn=generate_response,
        inputs=[chatbot, model_selector, tools_toggle, tts_toggle],
        outputs=[chatbot, audio_output]
    )
    
    # 清空：history 变 []，音频清空
    clear_btn.click(
        fn=lambda: ([], None),
        outputs=[chatbot, audio_output]
    )

# 启动完整版 UI
demo_full.launch()
